In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "transformers==4.52.4",
    "accelerate==1.7.0",
    "datasets==3.6.0",
    "evaluate==0.4.3",
    "tokenizers==0.21.1"
])

0

In [1]:
import sys
import transformers
import accelerate
import datasets
import torch

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Transformers: 4.52.4
Accelerate: 1.7.0
Datasets: 3.6.0
Torch: 2.7.1+cu126
CUDA available: True


In [11]:
# =========================================================
# 1. IMPORTS AND DRIVE SETUP
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch import nn
from datasets import load_dataset, Dataset

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr


In [ ]:
 
TEST_SENTENCE_LOG = "/content/drive/MyDrive/RoBERTa_SingleStep_test_sentence_log.csv"

os.makedirs("/content/drive/MyDrive", exist_ok=True)

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow(["model", "roberta-base"])
    writer.writerow(["learning_rate", 2e-5])
    writer.writerow(["train_batch_size", 16])
    writer.writerow(["eval_batch_size", 16])
    writer.writerow(["epochs", 4])
    writer.writerow(["early_stopping_metric", "eval_loss"])
    writer.writerow(["early_stopping_patience", 0])
    writer.writerow([])


# =========================================================
# 2. SEED SETUP
# =========================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# 3. LOAD BRIGHTER DATASET
# =========================================================
print("\nLoading BRIGHTER dataset from Hugging Face...")

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("\nOriginal split sizes:")
print("Train:", len(train_df))
print("Dev  :", len(val_df))
print("Test :", len(test_df))

print("\nSample data:")
print(train_df.head())


# =========================================================
# 4. LABEL CONFIGURATION
# =========================================================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
LEVELS = [1, 2, 3]

LABELS = [f"{emotion}_{level}" for emotion in EMOTIONS for level in LEVELS]
NUM_LABELS = len(LABELS)

print("\nLabels:")
print(LABELS)
print("Number of labels:", NUM_LABELS)


# =========================================================
# 5. CONVERT LABELS TO SINGLE-STEP FORMAT
# =========================================================
def convert_single_step_labels(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_1"] = (df[emotion] == 1).astype(int)
        df[f"{emotion}_2"] = (df[emotion] == 2).astype(int)
        df[f"{emotion}_3"] = (df[emotion] == 3).astype(int)

    return df


train_single = convert_single_step_labels(train_df)
val_single = convert_single_step_labels(val_df)
test_single = convert_single_step_labels(test_df)

print("\nSingle-step sample:")
print(train_single[["text"] + LABELS].head())


# =========================================================
# 6. COMBINE AND REDISTRIBUTE DATASET: 70 / 20 / 10
# =========================================================
full_df = pd.concat(
    [train_single, val_single, test_single],
    ignore_index=True
)

full_df = full_df[["text"] + LABELS]

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

n = len(full_df)

train_end = int(0.7 * n)
val_end = int(0.9 * n)

train_single = full_df[:train_end].reset_index(drop=True)
val_single = full_df[train_end:val_end].reset_index(drop=True)
test_single = full_df[val_end:].reset_index(drop=True)

print("\nRedistributed split sizes:")
print({
    "train": len(train_single),
    "val": len(val_single),
    "test": len(test_single)
})

# Keep original sentences before converting to Hugging Face Dataset
train_texts = train_single["text"].tolist()
val_texts = val_single["text"].tolist()
test_texts = test_single["text"].tolist()


# =========================================================
# 7. CONVERT TO HUGGING FACE DATASET
# =========================================================
train_single = Dataset.from_pandas(train_single, preserve_index=False)
val_single = Dataset.from_pandas(val_single, preserve_index=False)
test_single = Dataset.from_pandas(test_single, preserve_index=False)


# =========================================================
# 8. TOKENIZATION
# =========================================================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_single = train_single.map(tokenize_function, batched=True)
val_single = val_single.map(tokenize_function, batched=True)
test_single = test_single.map(tokenize_function, batched=True)


# =========================================================
# 9. ADD LABEL VECTOR
# =========================================================
def add_labels(example):
    example["labels"] = [float(example[label]) for label in LABELS]
    return example

train_single = train_single.map(add_labels)
val_single = val_single.map(add_labels)
test_single = test_single.map(add_labels)

train_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda

Loading BRIGHTER dataset from Hugging Face...

Original split sizes:
Train: 2763
Dev  : 115
Test : 2765

Sample data:
                        id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  disgust  fear  joy  sadness  surprise  
0      0      NaN     1    0        0         1  
1      0      NaN     2    0        0         0  
2      0      NaN     1    0        3         0  
3      0      NaN     0    0        0         

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

In [13]:
# =========================================================
# 10. METRICS
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(probs[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(labels[:, i], probs[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    pearson_mean = float(np.mean(pearsons))

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }


# =========================================================
# 11. LISTS FOR PLOTTING AND LOGGING
# =========================================================
epoch_list = []

train_loss_list = []

val_loss_list = []
val_f1_macro_list = []
val_f1_micro_list = []
val_pearson_mean_list = []

test_loss_list = []
test_f1_macro_list = []
test_f1_micro_list = []
test_pearson_mean_list = []


# =========================================================
# 12. CALLBACK FOR EPOCH LOGGING
# =========================================================
class SaveEpochResultsCallback(TrainerCallback):
    def __init__(self, file_path, test_dataset):
        self.file_path = file_path
        self.test_dataset = test_dataset

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if self._inside_eval:
            return

        if metrics is None:
            return

        self._inside_eval = True

        epoch = int(round(float(metrics.get("epoch", state.epoch))))

        train_loss = (
            self.current_train_loss
            if self.current_train_loss is not None
            else ""
        )

        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        test_results = self.trainer_ref.evaluate(
            eval_dataset=self.test_dataset,
            metric_key_prefix="test"
        )

        test_loss = float(test_results.get("test_loss", 0.0))
        test_f1_macro = float(test_results.get("test_f1_macro", 0.0))
        test_f1_micro = float(test_results.get("test_f1_micro", 0.0))
        test_pearson_mean = float(test_results.get("test_pearson_mean", 0.0))

        epoch_list.append(epoch)

        train_loss_list.append(train_loss)

        val_loss_list.append(val_loss)
        val_f1_macro_list.append(val_f1_macro)
        val_f1_micro_list.append(val_f1_micro)
        val_pearson_mean_list.append(val_pearson_mean)

        test_loss_list.append(test_loss)
        test_f1_macro_list.append(test_f1_macro)
        test_f1_micro_list.append(test_f1_micro)
        test_pearson_mean_list.append(test_pearson_mean)

        pred = self.trainer_ref.predict(self.test_dataset)

        logits = pred.predictions
        true_labels = pred.label_ids

        probs = 1 / (1 + np.exp(-logits))
        pred_labels = (probs >= 0.5).astype(int)

        report_dict = classification_report(
            true_labels,
            pred_labels,
            target_names=LABELS,
            zero_division=0,
            output_dict=True
        )

        with open(self.file_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([])
            writer.writerow([f"EPOCH {epoch}"])

            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "test_loss",
                "val_f1_macro",
                "val_f1_micro",
                "test_f1_macro",
                "test_f1_micro",
                "val_pearson_mean",
                "test_pearson_mean"
            ])

            for i in range(len(epoch_list)):
                writer.writerow([
                    epoch_list[i],
                    train_loss_list[i],
                    val_loss_list[i],
                    test_loss_list[i],
                    val_f1_macro_list[i],
                    val_f1_micro_list[i],
                    test_f1_macro_list[i],
                    test_f1_micro_list[i],
                    val_pearson_mean_list[i],
                    test_pearson_mean_list[i]
                ])

            # -------------------------
            # FINAL TEST SCORES AFTER THIS EPOCH
            # -------------------------
            writer.writerow([])
            writer.writerow([f"FINAL TEST SCORES AFTER EPOCH {epoch}"])
            writer.writerow(["metric", "value"])
            writer.writerow(["test_loss", test_loss])
            writer.writerow(["test_f1_macro", test_f1_macro])
            writer.writerow(["test_f1_micro", test_f1_micro])
            writer.writerow(["test_pearson_mean", test_pearson_mean])

            # -------------------------
            # CLASSWISE RESULTS AFTER THIS EPOCH
            # -------------------------
            writer.writerow([])
            writer.writerow([f"CLASSWISE RESULTS AFTER EPOCH {epoch}"])
            writer.writerow([
                "class",
                "precision",
                "recall",
                "f1_score",
                "support"
            ])

            for class_name in LABELS:
                row = report_dict.get(class_name, {})

                writer.writerow([
                    class_name,
                    row.get("precision", ""),
                    row.get("recall", ""),
                    row.get("f1-score", ""),
                    row.get("support", "")
                ])

        print(
            f"\nEpoch {epoch} cumulative losses, final test scores, "
            f"and classwise results saved."
        )

        self._inside_eval = False


# class EarlyStoppingWithMinEpoch(EarlyStoppingCallback):
#     def __init__(
#         self,
#         early_stopping_patience=0,
#         early_stopping_threshold=0.0,
#         min_epochs=2
#     ):
#         super().__init__(
#             early_stopping_patience=early_stopping_patience,
#             early_stopping_threshold=early_stopping_threshold
#         )
#         self.min_epochs = min_epochs

#     def on_evaluate(self, args, state, control, metrics=None, **kwargs):
#         current_epoch = state.epoch if state.epoch is not None else 0

#         if current_epoch < self.min_epochs:
#             return control

#         return super().on_evaluate(
#             args,
#             state,
#             control,
#             metrics=metrics,
#             **kwargs
#         )




# =========================================================
# 13. MODEL
# =========================================================
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)


# =========================================================
# 14. TRAINING ARGUMENTS WITH EARLY STOPPING ON EVAL LOSS
# =========================================================
training_args = TrainingArguments(
    output_dir="/content/roberta_output",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    # Early stopping monitors validation loss
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# =========================================================
# 15. TRAINER
# =========================================================
callback = SaveEpochResultsCallback(
    file_path=LOG_FILE,
    test_dataset=test_single
)

# early_stopping_callback = EarlyStoppingWithMinEpoch(
#     early_stopping_patience=0,
#     early_stopping_threshold=0.0,
#     min_epochs=2
# )


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_single,
    eval_dataset=val_single,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[callback]
)

callback.trainer_ref = trainer


# =========================================================
# 16. TRAIN
# =========================================================
start = time.time()

trainer.train()

end = time.time()

print(f"\nTotal training time: {end - start:.1f} seconds")
print("Epochwise result file saved at:", LOG_FILE)


# =========================================================
# 17. SENTENCE-WISE CLASSIFICATION LOGGING ONLY
# =========================================================
def clean_number(x):
    """
    Removes unnecessary trailing zeros.
    Example:
    0.0000 -> 0
    0.1500 -> 0.15
    1.0000 -> 1
    """
    x = round(float(x), 4)

    if x == 0:
        return 0

    if x == 1:
        return 1

    return x


def labels_to_text(binary_labels, label_names):
    selected_labels = [
        label_names[i]
        for i, value in enumerate(binary_labels)
        if int(value) == 1
    ]

    if len(selected_labels) == 0:
        return "No Emotion"

    return ", ".join(selected_labels)


def create_sentence_classification_log(
    trainer,
    dataset,
    raw_texts,
    label_names,
    output_path,
    threshold=0.5
):
    """
    Creates sentence-wise classification log:
    sentence, true labels, predicted labels, and class probabilities.
    """

    predictions = trainer.predict(dataset)

    logits = predictions.predictions
    y_true = predictions.label_ids

    probabilities = 1 / (1 + np.exp(-logits))
    y_pred = (probabilities >= threshold).astype(int)

    rows = []

    for i in range(len(raw_texts)):
        row = {
            "sentence_id": i,
            "sentence": raw_texts[i],
            "true_labels": labels_to_text(y_true[i], label_names),
            "predicted_labels": labels_to_text(y_pred[i], label_names)
        }

        for j, label in enumerate(label_names):
            row[f"prob_{label}"] = clean_number(probabilities[i][j])

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Saved sentence-wise classification log: {output_path}")

    return df


# =========================================================
# 18. SAVE SENTENCE-WISE LOGS FOR TRAIN, VAL, AND TEST
# =========================================================
train_sentence_log_df = create_sentence_classification_log(
    trainer=trainer,
    dataset=train_single,
    raw_texts=train_texts,
    label_names=LABELS,
    output_path=TRAIN_SENTENCE_LOG,
    threshold=0.5
)

val_sentence_log_df = create_sentence_classification_log(
    trainer=trainer,
    dataset=val_single,
    raw_texts=val_texts,
    label_names=LABELS,
    output_path=VAL_SENTENCE_LOG,
    threshold=0.5
)

test_sentence_log_df = create_sentence_classification_log(
    trainer=trainer,
    dataset=test_single,
    raw_texts=test_texts,
    label_names=LABELS,
    output_path=TEST_SENTENCE_LOG,
    threshold=0.5
)

print("\nSentence-wise train log saved at:", TRAIN_SENTENCE_LOG)
print("Sentence-wise validation log saved at:", VAL_SENTENCE_LOG)
print("Sentence-wise test log saved at:", TEST_SENTENCE_LOG)

print("\nSample train sentence-wise classification log:")
display(train_sentence_log_df.head(10))

print("\nSample validation sentence-wise classification log:")
display(val_sentence_log_df.head(10))

print("\nSample test sentence-wise classification log:")
display(test_sentence_log_df.head(10))


# =========================================================
# 19. FINAL TEST EVALUATION
# =========================================================
final_test_results = trainer.evaluate(
    eval_dataset=test_single,
    metric_key_prefix="final_test"
)

print("\nFinal test results:")
print(final_test_results)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Pearson Mean
1,0.328200,0.290549,0.000000,0.000000,0.195082
2,0.271500,0.255705,0.014208,0.018038,0.356362
3,0.240400,0.244332,0.103017,0.178453,0.402172
4,0.223100,0.240074,0.133344,0.233789,0.416213



Epoch 1 cumulative losses, final test scores, and classwise results saved.

Epoch 2 cumulative losses, final test scores, and classwise results saved.

Epoch 3 cumulative losses, final test scores, and classwise results saved.

Epoch 4 cumulative losses, final test scores, and classwise results saved.

Total training time: 166.9 seconds
Epochwise result file saved at: /content/drive/MyDrive/RoBERTa_SingleStep_log.csv


Saved sentence-wise classification log: /content/drive/MyDrive/RoBERTa_SingleStep_train_sentence_log.csv


Saved sentence-wise classification log: /content/drive/MyDrive/RoBERTa_SingleStep_val_sentence_log.csv


...


Sample validation sentence-wise classification log:


,sentence_id,sentence,true_labels,predicted_labels,prob_anger_1,prob_anger_2,prob_anger_3,prob_fear_1,prob_fear_2,prob_fear_3,prob_joy_1,prob_joy_2,prob_joy_3,prob_sadness_1,prob_sadness_2,prob_sadness_3,prob_surprise_1,prob_surprise_2,prob_surprise_3
0,0,"my throat is tight, my nose is numb.","fear_2, sadness_1",No Emotion,0.0463,0.0264,0.0191,0.4819,0.3511,0.0517,0.0434,0.0198,0.0113,0.4107,0.1557,0.0462,0.0344,0.0205,0.0146
1,1,"I went up on stage and sang my heart out, and ...",joy_2,No Emotion,0.0355,0.0294,0.0285,0.0912,0.0368,0.0337,0.3964,0.4298,0.1628,0.0627,0.0386,0.0429,0.1216,0.0475,0.0295
2,2,Her face when her mum told her she miscarried ...,"fear_1, sadness_3",No Emotion,0.1126,0.0774,0.0720,0.3769,0.2330,0.0898,0.0523,0.0378,0.0295,0.2725,0.3943,0.4853,0.0876,0.0360,0.0299
3,3,... but my head still throbs because eventuall...,"fear_1, sadness_1",fear_2,0.0606,0.0349,0.0249,0.3888,0.5047,0.1025,0.0302,0.0193,0.0128,0.4366,0.2006,0.0689,0.0413,0.0225,0.0172
4,4,My ears bent back.,surprise_1,No Emotion,0.0261,0.0160,0.0098,0.3841,0.0945,0.0212,0.1142,0.0226,0.0098,0.1607,0.0434,0.0201,0.0592,0.0218,0.0094
5,5,I go to her pictures and see the image of what...,"joy_1, surprise_1",No Emotion,0.0488,0.0228,0.0107,0.4789,0.1448,0.0256,0.0919,0.0264,0.0117,0.1152,0.0342,0.0156,0.4022,0.1091,0.0170
6,6,"Dropping the knife, and emptying my mouth, I p...",joy_1,No Emotion,0.1992,0.1084,0.0483,0.4087,0.3276,0.0666,0.0378,0.0153,0.0143,0.4244,0.2222,0.1387,0.0667,0.0217,0.0195
7,7,"""The name of my rig is Phantom 309.",surprise_1,No Emotion,0.0269,0.0197,0.0115,0.1862,0.0396,0.0222,0.3223,0.0712,0.0219,0.0519,0.0220,0.0169,0.2898,0.0458,0.0147
8,8,its 11:30 pm in krakow and i can hardly open m...,"fear_1, sadness_1",No Emotion,0.0405,0.0264,0.0157,0.2320,0.4260,0.1588,0.0353,0.0228,0.0132,0.2557,0.0953,0.0412,0.0804,0.0392,0.0169
9,9,The entire summer is going to pass by with a b...,"fear_1, joy_1",joy_1,0.0288,0.0234,0.0213,0.1469,0.0307,0.0232,0.5187,0.1961,0.0516,0.0667,0.0339,0.0312,0.1036,0.0328,0.0190



Sample test sentence-wise classification log:


,sentence_id,sentence,true_labels,predicted_labels,prob_anger_1,prob_anger_2,prob_anger_3,prob_fear_1,prob_fear_2,prob_fear_3,prob_joy_1,prob_joy_2,prob_joy_3,prob_sadness_1,prob_sadness_2,prob_sadness_3,prob_surprise_1,prob_surprise_2,prob_surprise_3
0,0,He changed our last name ever-so-slightly and ...,surprise_2,surprise_1,0.2287,0.0839,0.0347,0.4823,0.1669,0.0336,0.1018,0.0322,0.0216,0.1612,0.0720,0.0380,0.7339,0.2230,0.0366
1,1,They just never went away.,"fear_1, sadness_2",fear_1,0.2355,0.1176,0.0512,0.5139,0.1824,0.0327,0.0586,0.0197,0.0169,0.4042,0.2572,0.1419,0.1640,0.0382,0.0253
2,2,"""Well I have loads, you can share mine"".",joy_1,joy_1,0.0298,0.0237,0.0239,0.1047,0.0296,0.0285,0.5987,0.2346,0.0647,0.0615,0.0299,0.0310,0.1144,0.0396,0.0224
3,3,Pretty much everyone objected to my wedding.,"anger_1, fear_1, sadness_2, surprise_1",No Emotion,0.3410,0.1706,0.0479,0.3798,0.0860,0.0285,0.1339,0.0390,0.0291,0.2145,0.0848,0.0709,0.3440,0.0624,0.0287
4,4,"I was in better bike shape last year, and had ...",No Emotion,joy_1,0.0270,0.0234,0.0249,0.1045,0.0333,0.0283,0.5699,0.2728,0.0809,0.0669,0.0320,0.0344,0.0820,0.0371,0.0230
5,5,"Somehow, while scrubbing the left side of my f...","fear_1, surprise_1","fear_2, fear_3",0.1101,0.0677,0.0375,0.1472,0.5371,0.5194,0.0250,0.0243,0.0195,0.2748,0.1509,0.0920,0.2539,0.1241,0.0388
6,6,At about 4 i was running with a pole with one ...,"fear_2, surprise_1",fear_2,0.0836,0.0432,0.0178,0.3029,0.5367,0.2252,0.0323,0.0214,0.0143,0.2001,0.0609,0.0270,0.3907,0.2460,0.0353
7,7,No one else in the class did.,"sadness_1, surprise_1",No Emotion,0.1434,0.0533,0.0197,0.4240,0.1076,0.0234,0.0761,0.0191,0.0139,0.1790,0.0563,0.0321,0.4239,0.0624,0.0168
8,8,My tongue had sucked it up and the hole was cl...,surprise_1,No Emotion,0.0557,0.0265,0.0129,0.4875,0.0817,0.0157,0.1199,0.0223,0.0102,0.1886,0.0388,0.0186,0.1063,0.0284,0.0113
9,9,"Because this is a karma thread, you know how t...",surprise_1,surprise_1,0.2101,0.0848,0.0306,0.4488,0.0914,0.0235,0.1438,0.0309,0.0212,0.1514,0.0499,0.0296,0.5378,0.0885,0.0223



Epoch 4 cumulative losses, final test scores, and classwise results saved.

Final test results:
{'final_test_loss': 0.23544730246067047, 'final_test_f1_macro': 0.14293920242777364, 'final_test_f1_micro': 0.24825174825174826, 'final_test_pearson_mean': 0.39923718571662903, 'final_test_runtime': 0.6319, 'final_test_samples_per_second': 894.18, 'final_test_steps_per_second': 56.974, 'epoch': 4.0}
